# **Image Normalization Pipeline**

## **Setup and Installtions**

In [ ]:
# Install torchstain for Macenko normalization
!pip install torchstain
!pip install opencv-python-headless

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torchstain
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

# Set seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed(42)

## **Configuration**

In [ ]:
#Connection with GoogleDrive
from google.colab import drive
drive.mount('/gdrive')


# --- CONFIGURATION ---
BASE_DIR = '/gdrive/MyDrive/AN2DL_Chall2/AN2DL_Clean_DS'  # Point to your Cleaned Dataset
SOURCE_IMG_DIR = os.path.join(BASE_DIR, 'train_data')
NORM_IMG_DIR = os.path.join(BASE_DIR, 'train_data_normalized') # We will create this
# SOURCE_IMG_DIR = os.path.join(BASE_DIR, 'test_data')
# NORM_IMG_DIR = os.path.join(BASE_DIR, 'test_data_normalized') # We will create this

# Create the output directory for normalized images
os.makedirs(NORM_IMG_DIR, exist_ok=True)

print(f"Source: {SOURCE_IMG_DIR}")
print(f"Target: {NORM_IMG_DIR}")

In [ ]:
# python
import os

dir_path = SOURCE_IMG_DIR  # uses the variable already defined in the notebook

if not os.path.isdir(dir_path):
    print(f"Directory `NORM_IMG_DIR` does not exist: `{dir_path}`")
else:
    files = os.listdir(dir_path)
    img_count = sum(1 for f in files if f.startswith('img_') and os.path.isfile(os.path.join(dir_path, f)))
    mask_count = sum(1 for f in files if f.startswith('mask_') and os.path.isfile(os.path.join(dir_path, f)))

    print(f"Found {img_count} files starting with `img_` in `NORM_IMG_DIR` (`{dir_path}`)")
    print(f"Found {mask_count} files starting with `mask_` in `NORM_IMG_DIR` (`{dir_path}`)")


# **The Stain Normalizer**

In [ ]:
class StainNormalizer:
    def __init__(self, target_file=None):
        """
        Initialize Macenko normalizer.
        target_file: Path to a 'perfect' reference image. If None, uses internal defaults.
        """
        self.normalizer = torchstain.normalizers.MacenkoNormalizer(backend='numpy')

        # If you have a specific reference image you like, load it here.
        # Otherwise, we fit to a standard reference automatically.
        if target_file:
            target = cv2.cvtColor(cv2.imread(target_file), cv2.COLOR_BGR2RGB)
            self.normalizer.fit(target)

    def normalize(self, image):
        """
        Input: RGB numpy array (H, W, 3)
        Output: Normalized RGB numpy array
        """
        try:
            # torchstain expects RGB
            norm, H, E = self.normalizer.normalize(I=image, stains=False)
            return norm
        except Exception as e:
            # Fallback for images with no tissue (pure white) or singular matrices
            # print(f"Normalization failed: {e}")
            return image

# Initialize
normalizer = StainNormalizer()
print("✅ Stain Normalizer Ready")

## **Preprocessing Loop**

In [ ]:
def process_and_save(filename):
    """Worker function to process a single image"""
    if not filename.startswith('img_'):
        return

    # Paths
    src_path = os.path.join(SOURCE_IMG_DIR, filename)
    dst_path = os.path.join(NORM_IMG_DIR, filename)

    # Skip if already exists (resume capability)
    if os.path.exists(dst_path):
        return

    # Read
    img = cv2.imread(src_path)
    if img is None: return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Normalize
    img_norm = normalizer.normalize(img)

    # Save as PNG (lossless) - Convert back to BGR for OpenCV
    img_bgr = cv2.cvtColor(img_norm, cv2.COLOR_RGB2BGR)
    cv2.imwrite(dst_path, img_bgr)

# Get list of files
files = [f for f in os.listdir(SOURCE_IMG_DIR) if f.startswith('img_')]
print(f"Processing {len(files)} images...")

# Run in parallel to speed up IO
with ThreadPoolExecutor(max_workers=4) as executor:
    list(tqdm(executor.map(process_and_save, files), total=len(files), desc="Normalizing"))

print("✅ Offline Normalization Complete!")

In [ ]:
import os

dir_path = NORM_IMG_DIR  # uses the variable already defined in the notebook

if not os.path.isdir(dir_path):
    print(f"Directory `NORM_IMG_DIR` does not exist: `{dir_path}`")
else:
    files = os.listdir(dir_path)
    img_count = sum(1 for f in files if f.startswith('img_') and os.path.isfile(os.path.join(dir_path, f)))
    mask_count = sum(1 for f in files if f.startswith('mask_') and os.path.isfile(os.path.join(dir_path, f)))

    print(f"Found {img_count} files starting with `img_` in `NORM_IMG_DIR` (`{dir_path}`)")
    print(f"Found {mask_count} files starting with `mask_` in `NORM_IMG_DIR` (`{dir_path}`)")


In [ ]:
# Copy mask files from SOURCE_IMG_DIR to NORM_IMG_DIR if you want masks alongside normalized images
print("Copying masks...")
import shutil

for fname in os.listdir(SOURCE_IMG_DIR):
    if fname.startswith('mask_'):
        src = os.path.join(SOURCE_IMG_DIR, fname)
        dst = os.path.join(NORM_IMG_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

In [ ]:
import os

dir_path = NORM_IMG_DIR  # uses the variable already defined in the notebook

if not os.path.isdir(dir_path):
    print(f"Directory `NORM_IMG_DIR` does not exist: `{dir_path}`")
else:
    files = os.listdir(dir_path)
    img_count = sum(1 for f in files if f.startswith('img_') and os.path.isfile(os.path.join(dir_path, f)))
    mask_count = sum(1 for f in files if f.startswith('mask_') and os.path.isfile(os.path.join(dir_path, f)))

    print(f"Found {img_count} files starting with `img_` in `NORM_IMG_DIR` (`{dir_path}`)")
    print(f"Found {mask_count} files starting with `mask_` in `NORM_IMG_DIR` (`{dir_path}`)")


## **Pipeline Verification**

In [ ]:
class RobustTissueDataset(Dataset):
    def __init__(self, df, data_dir, mask_dir, transform=None,
                 patch_size=224,
                 patches_per_image=16,
                 mode='train'):
        """
        Args:
            df: DataFrame with 'sample_index'
            data_dir: Path to NORMALIZED images
            mask_dir: Path to ORIGINAL masks (masks are usually not normalized)
            transform: Albumentations transforms
            mode: 'train' or 'val'
        """
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.mask_dir = mask_dir  # Points to original train_data
        self.transform = transform
        self.patch_size = patch_size
        self.n_patches = patches_per_image
        self.mode = mode

        self.label_map = {'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['sample_index']
        label = self.label_map[row['label']]

        # 1. Load Image (Normalized)
        img_path = os.path.join(self.data_dir, img_name)
        image = cv2.imread(img_path)
        if image is None: # Safety check
            image = np.zeros((1000, 1000, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Load Mask (Original)
        # Masks usually have prefix 'mask_' instead of 'img_'
        mask_name = img_name.replace('img_', 'mask_')
        mask_path = os.path.join(self.mask_dir, mask_name)

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            # Threshold to be binary (0 or 255)
            _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        else:
            # Fallback: Create a mask of the whole image if missing
            mask = np.ones(image.shape[:2], dtype=np.uint8) * 255

        # 3. Smart Extraction
        if self.mode == 'train':
            patches = self._random_tumor_sampling(image, mask)
        else:
            patches = self._deterministic_grid_sampling(image, mask)

        # 4. Transform & Stack
        processed_patches = []
        for p in patches:
            if self.transform:
                p = self.transform(image=p)['image']
            else:
                p = torch.from_numpy(p.transpose(2, 0, 1)).float() / 255.0
            processed_patches.append(p)

        return torch.stack(processed_patches), label

    def _pad_patch(self, patch):
        """Reflection Padding to handle normalization artifacts."""
        h, w, c = patch.shape
        if h == self.patch_size and w == self.patch_size:
            return patch

        pad_h = self.patch_size - h
        pad_w = self.patch_size - w

        # Use REFLECT_101 to avoid "box artifacts"
        padded = cv2.copyMakeBorder(patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
        return padded

    def _random_tumor_sampling(self, image, mask):
        """
        TRAIN: Pick random points INSIDE the mask.
        """
        patches = []
        h, w = image.shape[:2]

        # Get coordinates where mask is white (Tumor)
        ys, xs = np.where(mask > 0)

        # If mask is empty (rare), fall back to random sampling
        if len(ys) == 0:
            ys = np.random.randint(0, h, size=100)
            xs = np.random.randint(0, w, size=100)

        for _ in range(self.n_patches):
            # Pick a random pixel from the tumor area
            idx = np.random.randint(len(ys))
            cy, cx = ys[idx], xs[idx]

            # Add small jitter so we don't always center exactly on a pixel
            cy = np.clip(cy + np.random.randint(-20, 20), 0, h)
            cx = np.clip(cx + np.random.randint(-20, 20), 0, w)

            # Calculate crop coordinates
            y1 = max(0, cy - self.patch_size // 2)
            x1 = max(0, cx - self.patch_size // 2)
            y2 = min(h, y1 + self.patch_size)
            x2 = min(w, x1 + self.patch_size)

            # Correct bounds if crop goes off-screen
            if y2 - y1 < self.patch_size: y1 = max(0, y2 - self.patch_size)
            if x2 - x1 < self.patch_size: x1 = max(0, x2 - self.patch_size)

            patch = image[y1:y2, x1:x2]
            patch = self._pad_patch(patch)
            patches.append(patch)

        return patches

    def _deterministic_grid_sampling(self, image, mask):
        """
        VAL: Grid search that prioritizes tiles with the most mask overlap.
        """
        patches = []
        h, w = image.shape[:2]
        candidates = []

        step = self.patch_size  # Non-overlapping grid

        for y in range(0, h, step):
            for x in range(0, w, step):
                y2 = min(h, y + self.patch_size)
                x2 = min(w, x + self.patch_size)

                # Check how much of this tile is inside the mask
                mask_patch = mask[y:y2, x:x2]
                tissue_pixels = np.sum(mask_patch > 0)

                candidates.append({
                    'y': y, 'x': x, 'h': y2-y, 'w': x2-x,
                    'score': tissue_pixels  # Higher score = more tumor
                })

        # Sort candidates by amount of tumor tissue (descending)
        candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)

        # Select top N patches
        selected = candidates[:self.n_patches]

        # If image is small and we don't have N patches, repeat the best ones
        while len(selected) < self.n_patches:
            selected.append(selected[0] if len(selected) > 0 else {'y':0,'x':0,'h':self.patch_size,'w':self.patch_size})

        for c in selected:
            patch = image[c['y']:c['y']+c['h'], c['x']:c['x']+c['w']]
            patch = self._pad_patch(patch)
            patches.append(patch)

        return patches


print("✅ RobustTissueDataset Defined")

In [ ]:
train_labels_df = pd.read_csv(os.path.join(BASE_DIR, 'train_labels.csv'))

ds = RobustTissueDataset(
    train_labels_df,
    NORM_IMG_DIR, # Use the normalized images
    SOURCE_IMG_DIR,
    patch_size=224,
    patches_per_image=16,
    mode='train'
)

# Get one item
patches, label = ds[43] # Returns Tensor of shape (4, 3, 256, 256)

print(patches.size())


# Visualization - 4x4 Grid
fig, axes = plt.subplots(4, 4, figsize=(12, 12))  # Changed to 4 rows, 4 cols, bigger square figure

# Flatten the 2D axes array to make iterating easier (0 to 15)
for i, ax in enumerate(axes.flat):
    if i < len(patches):
        # Convert tensor back to numpy image
        p = patches[i].permute(1, 2, 0).numpy()
        ax.imshow(p)
        ax.set_title(f"Patch {i+1}")

    # Hide axis ticks for a cleaner look
    ax.axis('off')

plt.suptitle(f"Label: {label} (16 Patches Grid)", fontsize=16)
plt.tight_layout()
plt.show()